In [1]:
from pathlib import Path
import numpy as np
import pandas as pd


DATA_DIR = Path("../../hcris-data/HCRIS_v1996/").resolve()
OUTPUT_DIR = Path("output").resolve() 

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


HCRIS_VARS = [
    # (name, wksht_cd, line_num, clmn_num, source)
    ("beds",                    "S300001", "01200", "0100", "numeric"),
    ("tot_charges",             "G300000", "00100", "0100", "numeric"),
    ("net_pat_rev",             "G300000", "00300", "0100", "numeric"),
    ("tot_discounts",           "G300000", "00200", "0100", "numeric"),
    ("tot_operating_exp",       "G300000", "00400", "0100", "numeric"),
    ("ip_charges",              "G200000", "00100", "0100", "numeric"),
    ("icu_charges",             "G200000", "01500", "0100", "numeric"),
    ("ancillary_charges",       "G200000", "01700", "0100", "numeric"),
    ("tot_discharges",          "S300001", "00100", "1500", "numeric"),
    ("mcare_discharges",        "S300001", "00100", "1300", "numeric"),
    ("mcaid_discharges",        "S300001", "00100", "1400", "numeric"),
    ("tot_mcare_payment",       "E00A18A", "01600", "0100", "numeric"),
    ("secondary_mcare_payment","E00A18A", "01700", "0100", "numeric"),
    ("street",                  "S200000", "00100", "0100", "alpha"),
    ("city",                    "S200000", "00101", "0100", "alpha"),
    ("state",                   "S200000", "00101", "0200", "alpha"),
    ("zip",                     "S200000", "00101", "0300", "alpha"),
    ("county",                  "S200000", "00101", "0400", "alpha"),
    ("name",                    "S200000", "00200", "0100", "alpha"),
    ("uncomp_care",             "S100000", "03000", "0100", "numeric"),
    ("cost_to_charge",          "S100000", "02400", "0100", "numeric"),
    ("new_cap_ass",             "A700002", "00900", "0200", "numeric"),
    ("cash",                    "G000000", "00100", "0100", "numeric"),
    ("fixed_assets",            "G000000", "02100", "0100", "numeric"),
    ("depr_land",               "G000000", "01301", "0100", "numeric"),
    ("depr_bldg",               "G000000", "01401", "0100", "numeric"),
    ("depr_lease",              "G000000", "01501", "0100", "numeric"),
    ("depr_fixed_equip",        "G000000", "01601", "0100", "numeric"),
    ("depr_auto",               "G000000", "01701", "0100", "numeric"),
    ("depr_major_equip",        "G000000", "01801", "0100", "numeric"),
    ("depr_minor_equip",        "G000000", "01901", "0100", "numeric"),
    ("current_assets",          "G000000", "01100", "0100", "numeric"),
    ("current_liabilities",     "G000000", "03600", "0100", "numeric"),
    ("pps_ip_charges",          "C000001", "10100", "0600", "numeric"),
    ("pps_op_charges",          "C000001", "10100", "0700", "numeric"),
    ("pps_mcare_cost",          "D10A181", "04900", "0100", "numeric"),
    ("pps_pgm_cost",            "D10A181", "05300", "0100", "numeric"),
]

RPT_COLS = [
    "RPT_REC_NUM", "PRVDR_CTRL_TYPE_CD", "PRVDR_NUM", "NPI",
    "RPT_STUS_CD", "FY_BGN_DT", "FY_END_DT", "PROC_DT",
    "INITL_RPT_SW", "LAST_RPT_SW", "TRNSMTL_NUM", "FI_NUM",
    "ADR_VNDR_CD", "FI_CREAT_DT", "UTIL_CD", "NPR_DT",
    "SPEC_IND", "FI_RCPT_DT",
]

LONG_COLS = ["RPT_REC_NUM", "WKSHT_CD", "LINE_NUM", "CLMN_NUM", "ITM_VAL_NUM"]

# Extraction ---------------------------------------------------------------

def extract_v1996():
    print(f"Extracting HCRIS v1996 (1998-2011) from: {DATA_DIR}")
    frames = []
    
    for yr in range(1998, 2012):
        # Adjusted path logic to look inside the HCRIS_v1996 folder
        base = DATA_DIR / f"HospitalFY{yr}"
        
        # Check if directory exists before proceeding
        if not base.exists():
            print(f"  Warning: Directory {base} not found. Skipping year {yr}.")
            continue

        try:
            alpha = pd.read_csv(base / f"hosp_{yr}_ALPHA.CSV", header=None, names=LONG_COLS, dtype=str)
            nmrc = pd.read_csv(base / f"hosp_{yr}_NMRC.CSV", header=None, names=LONG_COLS, dtype=str)
            rpt = pd.read_csv(base / f"hosp_{yr}_RPT.CSV", header=None, names=RPT_COLS, dtype=str)

            final = rpt[["RPT_REC_NUM", "PRVDR_NUM", "NPI", "FY_BGN_DT", "FY_END_DT",
                          "PROC_DT", "FI_CREAT_DT", "RPT_STUS_CD"]].copy()
            final.columns = ["report", "provider_number", "npi", "fy_start", "fy_end",
                             "date_processed", "date_created", "status"]
            final["year"] = yr
            final["data_source"] = "v1996"

            for var_name, wksht, line, col, source in HCRIS_VARS:
                data = alpha if source == "alpha" else nmrc
                
                # Filtering and merging logic
                vals = (
                    data
                    .loc[(data["WKSHT_CD"] == wksht) &
                         (data["LINE_NUM"] == line) &
                         (data["CLMN_NUM"] == col),
                         ["RPT_REC_NUM", "ITM_VAL_NUM"]]
                    .rename(columns={"RPT_REC_NUM": "report", "ITM_VAL_NUM": var_name})
                )
                
                if source == "numeric":
                    vals[var_name] = pd.to_numeric(vals[var_name], errors="coerce")
                
                # Ensure 'report' columns are the same type for the merge
                vals["report"] = vals["report"].astype(str)
                final["report"] = final["report"].astype(str)
                
                final = final.merge(vals, on="report", how="left")

            frames.append(final)
            print(f"  v1996 {yr}: {len(final):,} reports")
            
        except FileNotFoundError as e:
            print(f"  Error loading files for {yr}: {e}")

    if frames:
        combined = pd.concat(frames, ignore_index=True)
        output_path = OUTPUT_DIR / "HCRIS_Data_v1996.txt"
        combined.to_csv(output_path, sep="\t", index=False)
        print(f"  v1996 total: {len(combined):,} rows written to {output_path}")
        return combined
    else:
        print("No data extracted.")
        return None


if __name__ == "__main__":
    extract_v1996()

Extracting HCRIS v1996 (1998-2011) from: /scion/5261/econ470001/hcris-data/HCRIS_v1996
  v1996 1998: 6,327 reports
  v1996 1999: 6,210 reports
  v1996 2000: 6,195 reports
  v1996 2001: 6,172 reports
  v1996 2002: 6,198 reports
  v1996 2003: 6,193 reports
  v1996 2004: 6,265 reports
  v1996 2005: 6,248 reports
  v1996 2006: 6,233 reports
  v1996 2007: 6,180 reports
  v1996 2008: 6,206 reports
  v1996 2009: 6,196 reports
  v1996 2010: 3,851 reports
  v1996 2011: 34 reports
  v1996 total: 78,508 rows written to /home/ikazani/econ470/a0/work/hwk4/data/output/HCRIS_Data_v1996.txt
